# K-means: теория, реализация с нуля и практика

K-means — алгоритм **обучения без учителя**: у объектов нет известных классов, а модель разбивает их на $K$ групп по близости в пространстве признаков.

**Цель ноутбука:** понять целевую функцию, вывести два шага алгоритма, реализовать K-means с нуля, разобраться с инициализацией, масштабированием и выбором числа кластеров.

## Содержание
1. [Постановка задачи](#1-постановка-задачи)
2. [Почему центр кластера — среднее](#2-почему-центр-кластера--среднее)
3. [Алгоритм Ллойда](#3-алгоритм-ллойда)
4. [Реализация с нуля](#4-реализация-с-нуля)
5. [Инициализация и k-means++](#5-инициализация-и-k-means)
6. [Масштабирование признаков](#6-масштабирование-признаков)
7. [Как выбрать число кластеров](#7-как-выбрать-число-кластеров)
8. [KMeans из scikit-learn](#8-kmeans-из-scikit-learn)
9. [Пример на Iris](#9-пример-на-iris)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.datasets import load_iris, make_blobs
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 1. Постановка задачи

Пусть даны объекты $X=\{x_1,\ldots,x_n\}$, где $x_i\in\mathbb R^d$. Требуется выбрать $K$ центров $\mu_1,\ldots,\mu_K$ и назначить каждому объекту номер кластера $c_i\in\{1,\ldots,K\}$.

K-means минимизирует сумму квадратов расстояний от объектов до центров своих кластеров:

$$J(C,\mu)=\sum_{i=1}^{n}\left\|x_i-\mu_{c_i}\right\|_2^2.
$$

В `scikit-learn` это значение называется **inertia**. Иногда используют среднее $J/n$: минимум не изменится, потому что множитель $1/n$ постоянен.

Целевая функция одновременно зависит от дискретных назначений $C$ и непрерывных центров $\mu$. Перебрать все $K^n$ назначений невозможно уже для умеренного $n$, поэтому K-means поочерёдно оптимизирует одну группу переменных при фиксированной другой.

## 2. Почему центр кластера — среднее

Зафиксируем объекты одного кластера $S$ и найдём центр $\mu$, минимизирующий

$$J_S(\mu)=\sum_{x_i\in S}\|x_i-\mu\|_2^2.
$$

Раскроем производную покоординатно:

$$\nabla_\mu J_S=2\sum_{x_i\in S}(\mu-x_i).$$

Приравнивая её к нулю, получаем

$$|S|\mu=\sum_{x_i\in S}x_i,\qquad\mu=\frac1{|S|}\sum_{x_i\in S}x_i.
$$

Именно квадрат евклидова расстояния приводит к **среднему**. Для суммы обычных евклидовых расстояний оптимален геометрический медианный центр, а для суммы $L_1$-расстояний — покоординатная медиана. Поэтому произвольная замена метрики ломает математическое обоснование стандартного шага K-means.

## 3. Алгоритм Ллойда

После выбора начальных центров повторяются два шага:

1. **Assignment:** назначить объект ближайшему центру:
   $$c_i=\arg\min_k\|x_i-\mu_k\|_2^2.$$
2. **Update:** заменить каждый центр средним назначенных ему объектов:
   $$\mu_k=\frac{1}{|C_k|}\sum_{i:c_i=k}x_i.$$

Assignment минимизирует $J$ при фиксированных центрах, update — при фиксированных назначениях. Поэтому inertia на итерациях **не возрастает**. Возможных разбиений конечное число, следовательно алгоритм остановится, когда назначения перестанут меняться либо сдвиг центров станет достаточно мал.

Но гарантируется только **локальный минимум**. Разные начальные центры могут привести к разным решениям, поэтому используют несколько запусков (`n_init`) и оставляют результат с минимальной inertia.

Грубая стоимость одной итерации — $O(nKd)$: нужно сравнить $n$ объектов с $K$ центрами в $d$ измерениях. Для $T$ итераций стоимость порядка $O(TnKd)$.

In [ ]:
X_demo, y_demo = make_blobs(
    n_samples=300, centers=3, cluster_std=[0.75, 0.9, 0.7],
    random_state=RANDOM_STATE,
)

plt.figure(figsize=(6, 4))
plt.scatter(X_demo[:, 0], X_demo[:, 1], s=22, alpha=0.75)
plt.title('Данные без известных алгоритму меток')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.show()

## 4. Реализация с нуля

Ниже — учебная реализация алгоритма Ллойда. Она обрабатывает пустой кластер переносом его центра на самый далёкий от текущих центров объект. Промышленные реализации используют дополнительные оптимизации и более тщательно обрабатывают численные крайние случаи.

In [ ]:
class KMeansFromScratch:
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, random_state=None):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

    @staticmethod
    def _squared_distances(X, centers):
        return ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or len(X) < self.n_clusters:
            raise ValueError('X должен быть двумерным, а объектов должно быть не меньше K')

        rng = np.random.default_rng(self.random_state)
        indices = rng.choice(len(X), size=self.n_clusters, replace=False)
        centers = X[indices].copy()
        self.inertia_history_ = []

        for iteration in range(1, self.max_iter + 1):
            distances = self._squared_distances(X, centers)
            labels = distances.argmin(axis=1)
            new_centers = np.empty_like(centers)

            for cluster in range(self.n_clusters):
                members = X[labels == cluster]
                if len(members):
                    new_centers[cluster] = members.mean(axis=0)
                else:
                    nearest_distance = distances.min(axis=1)
                    new_centers[cluster] = X[nearest_distance.argmax()]

            new_distances = self._squared_distances(X, new_centers)
            new_labels = new_distances.argmin(axis=1)
            inertia = new_distances[np.arange(len(X)), new_labels].sum()
            self.inertia_history_.append(float(inertia))

            shift = np.linalg.norm(new_centers - centers)
            centers = new_centers
            if shift <= self.tol:
                break

        self.cluster_centers_ = centers
        final_distances = self._squared_distances(X, centers)
        self.labels_ = final_distances.argmin(axis=1)
        self.inertia_ = float(final_distances[np.arange(len(X)), self.labels_].sum())
        self.n_iter_ = iteration
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return self._squared_distances(X, self.cluster_centers_).argmin(axis=1)


scratch = KMeansFromScratch(n_clusters=3, random_state=4).fit(X_demo)
print(f'Итераций: {scratch.n_iter_}; inertia: {scratch.inertia_:.2f}')
print('Inertia не возрастает:', np.all(np.diff(scratch.inertia_history_) <= 1e-10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X_demo[:, 0], X_demo[:, 1], c=scratch.labels_, cmap='viridis', s=22)
axes[0].scatter(*scratch.cluster_centers_.T, c='red', marker='X', s=180, label='Центры')
axes[0].set_title('Кластеры реализации с нуля')
axes[0].legend()
axes[1].plot(np.arange(1, scratch.n_iter_ + 1), scratch.inertia_history_, marker='o')
axes[1].set(title='Монотонное уменьшение inertia', xlabel='Итерация', ylabel='Inertia')
plt.tight_layout()
plt.show()

## 5. Инициализация и k-means++

При случайном выборе центров несколько центров могут попасть в одну естественную группу, а другая останется без представителя. **k-means++** распределяет начальные центры по пространству:

1. Первый центр выбирается случайно.
2. Для каждого объекта считается $D(x)^2$ — квадрат расстояния до ближайшего выбранного центра.
3. Следующий центр выбирается с вероятностью, пропорциональной $D(x)^2$.
4. Шаги 2–3 повторяются до получения $K$ центров.

Далёкие объекты получают большую вероятность стать центром, но выбор остаётся случайным. Поэтому даже с k-means++ полезны несколько инициализаций. `random_state` обеспечивает воспроизводимость эксперимента, но не улучшает модель сам по себе.

In [ ]:
def kmeans_plusplus_init(X, n_clusters, random_state=None):
    X = np.asarray(X, dtype=float)
    rng = np.random.default_rng(random_state)
    centers = [X[rng.integers(len(X))]]

    while len(centers) < n_clusters:
        distances = ((X[:, None, :] - np.asarray(centers)[None, :, :]) ** 2).sum(axis=2)
        nearest_squared = distances.min(axis=1)
        probabilities = nearest_squared / nearest_squared.sum()
        centers.append(X[rng.choice(len(X), p=probabilities)])
    return np.asarray(centers)

initial_centers = kmeans_plusplus_init(X_demo, 3, RANDOM_STATE)
plt.figure(figsize=(6, 4))
plt.scatter(X_demo[:, 0], X_demo[:, 1], s=18, alpha=0.45)
plt.scatter(*initial_centers.T, c='red', marker='X', s=180, label='k-means++ центры')
plt.legend()
plt.title('Начальные центры распределены по пространству')
plt.show()

## 6. Масштабирование признаков

Евклидово расстояние зависит от единиц измерения. Если один признак изменяется от 0 до 100000, а другой — от 0 до 1, первый почти полностью определит кластеры. Это не техническая ошибка K-means: алгоритм честно оптимизирует заданную геометрию, но эта геометрия может не соответствовать смыслу задачи.

Частая отправная точка — `StandardScaler`. При сильных выбросах рассматривают `RobustScaler`, логарифмирование или отдельную обработку выбросов. Масштабирование следует обучать только на train, если кластеризация является частью более крупного predictive pipeline.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
X_scale, _ = make_blobs(n_samples=350, centers=3, cluster_std=0.75, random_state=7)
X_distorted = X_scale.copy()
X_distorted[:, 0] *= 100

raw_labels = KMeans(3, n_init=20, random_state=RANDOM_STATE).fit_predict(X_distorted)
scaled = StandardScaler().fit_transform(X_distorted)
scaled_labels = KMeans(3, n_init=20, random_state=RANDOM_STATE).fit_predict(scaled)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X_distorted[:, 0], X_distorted[:, 1], c=raw_labels, s=18, cmap='viridis')
axes[0].set_title('Без масштабирования')
axes[1].scatter(scaled[:, 0], scaled[:, 1], c=scaled_labels, s=18, cmap='viridis')
axes[1].set_title('После StandardScaler')
plt.tight_layout()
plt.show()

## 7. Как выбрать число кластеров

### Elbow method

Inertia всегда уменьшается при росте $K$ и станет нулевой при $K=n$. Поэтому выбирают не минимальное значение, а точку **локтя**, после которой добавление кластеров даёт небольшую дополнительную экономию. Локоть бывает субъективным или вообще незаметным.

### Silhouette score

Для объекта $i$ обозначим через $a(i)$ среднее расстояние до объектов своего кластера, а через $b(i)$ — минимальное среднее расстояние до объектов другого кластера. Тогда

$$s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}\in[-1,1].$$

Значение около 1 означает компактный и отделённый кластер, около 0 — объект на границе, отрицательное — возможное неверное назначение. Silhouette нельзя считать для $K=1$. Он тоже не является абсолютной истиной: критерий предпочитает компактные выпуклые группы и может не совпасть с предметным смыслом сегментов.

In [ ]:
k_values = range(2, 9)
inertias, silhouettes = [], []
for k in k_values:
    model = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_demo)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_demo, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(k_values), inertias, marker='o')
axes[0].set(title='Elbow method', xlabel='K', ylabel='Inertia')
axes[1].plot(list(k_values), silhouettes, marker='o')
axes[1].set(title='Silhouette score', xlabel='K', ylabel='Средний silhouette')
best_k = list(k_values)[int(np.argmax(silhouettes))]
axes[1].axvline(best_k, color='red', linestyle='--', label=f'лучший K={best_k}')
axes[1].legend()
plt.tight_layout()
plt.show()
print('Лучший K по silhouette:', best_k)

## 8. KMeans из scikit-learn

Основные параметры:

- `n_clusters` — число кластеров;
- `init='k-means++'` — способ инициализации;
- `n_init` — число независимых запусков;
- `max_iter` — предел итераций одного запуска;
- `tol` — допуск остановки по изменению центров;
- `random_state` — воспроизводимость.

Номера кластеров не имеют смысла порядка: кластер `0` не «меньше» кластера `1`. Между запусками номера могут переставиться, даже если само разбиение одинаково. Поэтому обычная accuracy против известных классов некорректна без сопоставления меток. Для внешней оценки используют permutation-invariant метрики, например Adjusted Rand Index (ARI).

In [ ]:
sk_model = KMeans(
    n_clusters=3, init='k-means++', n_init=20,
    max_iter=300, tol=1e-4, random_state=RANDOM_STATE,
)
sk_labels = sk_model.fit_predict(X_demo)

print(f'sklearn inertia: {sk_model.inertia_:.2f}')
print(f'реализация с нуля inertia: {scratch.inertia_:.2f}')
print(f'ARI между разбиениями: {adjusted_rand_score(sk_labels, scratch.labels_):.3f}')
print(f'Число итераций sklearn: {sk_model.n_iter_}')

## 9. Пример на Iris

У Iris известны настоящие виды, но при обучении K-means их не получает. Метки нужны только после кластеризации, чтобы оценить совпадение скрытой структуры с биологическими классами. Признаки стандартизируем: длины и ширины измерены в одинаковых единицах, но имеют разные диапазоны и дисперсии.

In [ ]:
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
iris_model = KMeans(n_clusters=3, n_init=30, random_state=RANDOM_STATE)
iris_clusters = iris_model.fit_predict(X_iris)

print(f'ARI с истинными видами: {adjusted_rand_score(iris.target, iris_clusters):.3f}')
print(f'Silhouette: {silhouette_score(X_iris, iris_clusters):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(iris.data[:, 2], iris.data[:, 3], c=iris.target, cmap='viridis', s=28)
axes[0].set_title('Истинные виды (не использовались в fit)')
axes[1].scatter(iris.data[:, 2], iris.data[:, 3], c=iris_clusters, cmap='viridis', s=28)
axes[1].set_title('Кластеры K-means')
for ax in axes:
    ax.set_xlabel('petal length')
    ax.set_ylabel('petal width')
plt.tight_layout()
plt.show()

Важно: кластер не обязан совпадать с классом. **Класс** задаётся внешней меткой и целью задачи, а **кластер** определяется выбранными признаками, масштабированием, расстоянием и целевой функцией. Один класс может состоять из нескольких кластеров, а один кластер — содержать несколько классов.